# 02 - Transform raw cache -> source-of-truth Parquet

Stage B of the phase-5 initial build. Reads the raw OMW + kaikki cache
(produced by `01_download`), transforms it into source-tagged
`concepts` / `lemmas` / `senses` Parquet under `data/lexicon/` (the source of
truth), writes the `_build.json` manifest, and carves a small sample slice.

Thin caller: all logic is in `ingestion.pipeline.build_initial`. After this runs,
`LexiconStore.from_data_fol` loads the full corpus and the `explore` notebook
under `notebooks/lexicon_corpus/` starts returning rows.


In [ ]:
from lang_tools.lexicon.ingestion.pipeline import build_initial
from lang_tools.params.lang_tools_params import get_lang_tools_params

LANGS = [
    "en",
    "pt",
    "es",
    "fr",
    "it",
]
data_fol = get_lang_tools_params().paths.data_fol
data_fol

## Build

Sources default to the raw cache under `data_fol` (so this needs the `ingest`
extra to read OMW via `wn`). Pass `extra_manifest=` the `omw_info` / `kaikki_info`
dicts from `01_download` to pin the source versions in the manifest.


In [ ]:
summary = build_initial(LANGS, data_fol=data_fol)
summary.counts, summary.sample_counts

## Spot-check: cross-lingual grouping + gloss coverage


In [ ]:
from lang_tools.lexicon.lemma_store import LexiconStore

store = LexiconStore.from_data_fol(data_fol)

house = next(lem for lem in store.get_lemmas_by_language("en") if lem.text == "house")
concept = store.concepts_for_lemma(house.id)[0]
print("definitions:", concept.definitions)
for lang in LANGS:
    forms = [lem.text for lem in store.lemmas_for_concept(concept.id, language=lang)]
    print(lang, forms)